In [ ]:
import os
import zipfile
import yaml
from ultralytics import YOLO
import os
import sys
import torch

ZIP_FILE = "dataset_clean-LV55.zip"

EXTRACT_FOLDER = "dataset_clean"
import shutil
if os.path.exists(EXTRACT_FOLDER):
    shutil.rmtree(EXTRACT_FOLDER)

# Extract
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall()

In [2]:
yaml_path=None

for root, dirs, files in os.walk(".", topdown=True):
    if "dataset.yaml" in files:
        yaml_path = os.path.join(root, "dataset.yaml")
        break

print("YAML FOUND AT:", yaml_path)

if yaml_path is None:
    raise Exception("dataset.yaml NOT FOUND ")

YAML FOUND AT: .\dataset.yaml


In [3]:
model =YOLO("yolov8n.pt")  #Best for Raspberry Pi...

In [4]:
#Finding dataset.yaml
yaml_path =None
for root, dirs, files in os.walk(".", topdown=True):
    if "dataset.yaml" in files:
        yaml_path = os.path.join(root, "dataset.yaml")
        break

print("Using YAML:", yaml_path)
#-----------------------------------------------------------------
#Loading YAML
with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)
#-----------------------------------------------------------------
#path to current folder
dataset_root = os.path.dirname(yaml_path)

data["path"] = dataset_root
data["train"] = "train/images"
data["val"]   = "val/images"
data["test"]  = "test/images"
#-----------------------------------------------------------------
#fixed YAML
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

Using YAML: .\dataset.yaml


In [6]:
results = model.train(
    data=yaml_path,
    epochs=20,
    imgsz=640,
   batch=16,          # Increased: GPU can likely handle more than 8
    workers=4,         # Increased: Faster image preprocessing
    device=0,          # Critical: Uses your NVIDIA GPU
    amp=True,          # Uses Mixed Precision for a speed boost
    name="drone_model"
    )

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=.\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drone_

In [7]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.31  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 252.075.6 MB/s, size: 28.0 KB)
val: Scanning C:\Users\USER\OneDrive\Desktop\DDTE system\NN\val\labels.cache... 322 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 322/322  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 6.7it/s 3.1s0.1s
                   all        322        350      0.946      0.931      0.956      0.563
Speed: 1.9ms preprocess, 4.2ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to C:\Users\USER\OneDrive\Desktop\DDTE system\NN\runs\detect\val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object

In [8]:
results = model.predict(
    source=yaml_path.replace("dataset.yaml", "test/images"),
    save=True
)

print("Predictions saved in runs/detect/predict/")

image 1/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\146.JPEG: 448x640 2 drones, 86.7ms
image 2/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\185.JPEG: 480x640 1 drone, 48.1ms
image 3/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\191.JPEG: 384x640 1 drone, 48.8ms
image 4/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\200.JPEG: 384x640 1 drone, 27.7ms
image 5/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\246.JPEG: 384x640 1 drone, 21.2ms
image 6/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\337.JPEG: 448x640 2 drones, 27.6ms
image 7/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\50.JPEG: 384x640 2 drones, 24.1ms
image 8/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\67.JPEG: 384x640 1 drone, 25.0ms
image 9/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\images\76.JPEG: 384x640 1 drone, 20.0ms
image 10/162 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\test\

In [9]:
model.export(format="onnx")
#this is for Pi

Ultralytics 8.4.31  Python-3.11.9 torch-2.5.1+cu121 CPU (13th Gen Intel Core i5-13420H)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/



PyTorch: starting from 'C:\Users\USER\OneDrive\Desktop\DDTE system\NN\runs\detect\drone_model2\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.4 MB 5.6 MB/s eta 0:00:03
   ------ --------------------------------- 2.6/16.4 MB 9.4 MB/s eta 0:00:02
   ------------ --------------------------- 5.2/16.4 MB 10.6 MB/s eta 0:00:02
   ------------------ --------------------- 7.6/16.4 MB 11.2 MB/s eta 0:00:01
   ------------------------ --------------- 10.2/16.4 MB 11.2 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.4 MB 11.4 MB/s eta 0:00:01
   ------------------------------------ --- 15.2/16.4 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16

'C:\\Users\\USER\\OneDrive\\Desktop\\DDTE system\\NN\\runs\\detect\\drone_model2\\weights\\best.onnx'

In [ ]:
print("BEST MODEL:")
print("runs/detect/drone_model/weights/best.pt")

print("\nONNX MODEL:")
print("runs/detect/drone_model/weights/best.onnx")99


#
#C:\Users\USER\OneDrive\Desktop\DDTE system\NN\runs\detect\drone_model2\weights
#

BEST MODEL:
runs/detect/drone_model/weights/best.pt

ONNX MODEL:
runs/detect/drone_model/weights/best.onnx


In [18]:
from ultralytics import YOLO
import cv2

# 1. Load your model using the full Windows path
# We use r"path" to handle the backslashes and spaces correctly
model_path = r"C:\Users\USER\OneDrive\Desktop\DDTE system\NN\runs\detect\drone_model2\weights\best.pt"
model = YOLO(model_path) 

# 2. Run inference on your photo
# Make sure drone_photo.png is in the same folder as this script, 
# or provide the full path like we did for the model.
image_path = "x.jpg" 
results = model.predict(source=image_path, save=True, conf=0.5)

# 3. Process the results
for result in results:
    # This will open a window and show the image with bounding boxes
    result.show()
    
    # Print how many drones were found
    print(f"Detected {len(result.boxes)} object(s) in the image.")

print(f"Detection complete. Results saved to: {results[0].save_dir}")


image 1/1 c:\Users\USER\OneDrive\Desktop\DDTE system\NN\x.jpg: 480x640 1 drone, 41.2ms
Speed: 24.5ms preprocess, 41.2ms inference, 11.3ms postprocess per image at shape (1, 3, 480, 640)
Results saved to C:\Users\USER\OneDrive\Desktop\DDTE system\NN\runs\detect\predict5
Detected 1 object(s) in the image.
Detection complete. Results saved to: C:\Users\USER\OneDrive\Desktop\DDTE system\NN\runs\detect\predict5
